In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
import io
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12
sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")

In [ ]:
import os
os.makedirs("news_category_visualization", exist_ok=True)
print("Directory created: news_category_visualization")

In [ ]:
CSV_PATH = "data/Sinhala-News-Category-classification/sinhala-news-categories.csv"
df = pd.read_csv(CSV_PATH)

# Normalize column names (strip whitespace)
df.columns = df.columns.str.strip()
df["comments"] = df["comments"].astype(str).str.strip()
df["labels"] = pd.to_numeric(df["labels"], errors="coerce")

print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
print(df.head())

In [ ]:
print("─" * 50)
print("DATAFRAME INFO")
print("─" * 50)
print(df.info())

print("\n─" * 50)
print("DESCRIPTIVE STATISTICS")
print("─" * 50)
print(df.describe(include="all"))

print("\n─" * 50)
print("MISSING VALUES")
print("─" * 50)
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing cells: {missing.sum()}")

print("\n─" * 50)
print("DUPLICATE ROWS")
print("─" * 50)
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

In [ ]:
label_counts = df["labels"].value_counts().sort_index()
label_pct = df["labels"].value_counts(normalize=True).sort_index() * 100

label_summary = pd.DataFrame({
    "Count": label_counts,
    "Percentage (%)": label_pct.round(2)
})
print("Label Distribution:")
print(label_summary)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
label_counts.plot(kind="bar", ax=axes[0], color=sns.color_palette("pastel"), edgecolor="black")
axes[0].set_title("Label Frequency")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)
for bar in axes[0].patches:
    axes[0].annotate(f"{int(bar.get_height())}",
                     (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     ha="center", va="bottom", fontsize=11)

# Pie chart
axes[1].pie(label_counts, labels=label_counts.index.astype(str),
            autopct="%1.1f%%", colors=sns.color_palette("pastel"),
            startangle=140, wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[1].set_title("Label Proportion")

plt.suptitle("Label Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("news_category_visualization/label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: news_category_visualization/label_distribution.png")

In [ ]:
df["char_count"] = df["comments"].str.len()
df["word_count"] = df["comments"].str.split().str.len()
df["unique_chars"] = df["comments"].apply(lambda x: len(set(x)))

print("Text Length Stats:")
print(df[["char_count", "word_count", "unique_chars"]].describe().round(2))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df["char_count"], bins=max(5, len(df)//2), color="#74b9ff", edgecolor="black")
axes[0].set_title("Character Count Distribution")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Frequency")

axes[1].hist(df["word_count"], bins=max(5, len(df)//2), color="#a29bfe", edgecolor="black")
axes[1].set_title("Word Count Distribution")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Frequency")

plt.suptitle("Text Length Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("news_category_visualization/text_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: news_category_visualization/text_length_distribution.png")

In [ ]:
print("Average character count per label:")
print(df.groupby("labels")["char_count"].mean().round(1))

print("\nAverage word count per label:")
print(df.groupby("labels")["word_count"].mean().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.groupby("labels")["char_count"].mean().plot(
    kind="bar", ax=axes[0], color=sns.color_palette("Set2"), edgecolor="black")
axes[0].set_title("Avg Char Count per Label")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Avg Characters")
axes[0].tick_params(axis="x", rotation=0)

df.groupby("labels")["word_count"].mean().plot(
    kind="bar", ax=axes[1], color=sns.color_palette("Set3"), edgecolor="black")
axes[1].set_title("Avg Word Count per Label")
axes[1].set_xlabel("Label")
axes[1].set_ylabel("Avg Words")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle("Text Length by Label", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("news_category_visualization/text_length_by_label.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: news_category_visualization/text_length_by_label.png")

In [ ]:
all_text = " ".join(df["comments"].tolist())
# Exclude spaces and punctuation for character frequency
char_freq = Counter(c for c in all_text if c.strip() and c.isalpha())
top_chars = char_freq.most_common(20)

chars, freqs = zip(*top_chars) if top_chars else ([], [])

plt.figure(figsize=(12, 5))
bars = plt.bar(range(len(chars)), freqs, color=sns.color_palette("husl", len(chars)))
plt.xticks(range(len(chars)), chars, fontsize=13)
plt.title("Top 20 Most Frequent Characters", fontsize=14, fontweight="bold")
plt.xlabel("Character")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("news_category_visualization/top_characters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: news_category_visualization/top_characters.png")

In [ ]:
print("=" * 55)
print("         FINAL DATASET SUMMARY")
print("=" * 55)
print(f"  Total samples          : {len(df)}")
print(f"  Unique labels          : {df['labels'].nunique()} → {sorted(df['labels'].dropna().unique().tolist())}")
print(f"  Missing comments       : {df['comments'].isnull().sum()}")
print(f"  Missing labels         : {df['labels'].isnull().sum()}")
print(f"  Avg comment length     : {df['char_count'].mean():.1f} chars")
print(f"  Avg word count         : {df['word_count'].mean():.1f} words")
print(f"  Shortest comment       : {df['char_count'].min()} chars")
print(f"  Longest comment        : {df['char_count'].max()} chars")
print("=" * 55)

print("\nSaved outputs:")
for fname in ["news_category_visualization/label_distribution.png", "news_category_visualization/text_length_distribution.png",
              "news_category_visualization/text_length_by_label.png", "news_category_visualization/top_characters.png"]:
    print(f"  • {fname}")

In [ ]:
def print_dataset_summary(df, text_col, label_col):
    total_samples = len(df)
    unique_labels = sorted(df[label_col].dropna().unique().tolist())
    missing_comments = df[text_col].isna().sum()
    missing_labels = df[label_col].isna().sum()
    
    # Safely compute lengths and word counts by filling NaNs with empty strings
    clean_text = df[text_col].fillna('')
    lengths = clean_text.str.len()
    word_counts = clean_text.str.split().str.len()
    
    # Note: We compute mean over non-null samples if you prefer, 
    # but filling with '' means missing comments are 0 chars/words.
    # To be precise, let's filter out original NaNs for averaging:
    non_null_mask = df[text_col].notna()
    avg_len = lengths[non_null_mask].mean() if non_null_mask.any() else 0
    avg_words = word_counts[non_null_mask].mean() if non_null_mask.any() else 0
    min_len = lengths[non_null_mask].min() if non_null_mask.any() else 0
    max_len = lengths[non_null_mask].max() if non_null_mask.any() else 0
    
    print('=======================================================')
    print('         FINAL DATASET SUMMARY')
    print('=======================================================')
    print(f'  Total samples          : {total_samples}')
    print(f'  Unique labels          : {len(unique_labels)} → {unique_labels}')
    print(f'  Missing comments       : {missing_comments}')
    print(f'  Missing labels         : {missing_labels}')
    print(f'  Avg comment length     : {avg_len:.1f} chars')
    print(f'  Avg word count         : {avg_words:.1f} words')
    print(f'  Shortest comment       : {min_len:.0f} chars')
    print(f'  Longest comment        : {max_len:.0f} chars')
    print('=======================================================')
    print("\nLabel Distribution:")
    
    counts = df[label_col].value_counts()
    pcts = (df[label_col].value_counts(normalize=True) * 100).round(2)
    dist_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts})
    print(dist_df.to_string())


## Sinhala Sentiment Analysis Dataset Summary


In [ ]:
# Note: Using train.csv as the base for summary
SENTIMENT_PATH = '../data/sinhala-sentiment-analysis/outputs/train.csv'
df_sent = pd.read_csv(SENTIMENT_PATH)
print_dataset_summary(df_sent, 'comment_phrase', 'comment_sentiment')

## Writing Style Classification Dataset Summary


In [ ]:
STYLE_PATH = '../data/Writing-style-classification/test/writing_style_test.csv'
df_style = pd.read_csv(STYLE_PATH)
print_dataset_summary(df_style, 'comments', 'labels')